In [11]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [12]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [13]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")

    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [14]:
dataset = generate_dataset()

dataset

[{'task': 'Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens only, must start and end with alphanumeric)'},
 {'task': 'Create a JSON CloudFormation template snippet that defines an AWS IAM role with a basic trust relationship policy for EC2 instances'},
 {'task': "Write a regular expression that matches valid AWS ARN (Amazon Resource Name) formats, such as 'arn:aws:s3:::bucket-name' or 'arn:aws:iam::123456789012:role/service-role'"}]

In [15]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [16]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
        """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [17]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [18]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case wich each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results


In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[{'output': '# AWS S3 Bucket Name Validation Regular Expression\n\nHere\'s a comprehensive solution with multiple approaches:\n\n## Solution 1: Single Regex Pattern (Recommended)\n\n```regex\n^(?!.*\\.\\.)(?!-)[a-z0-9.-]{3,63}(?<!-)(?<!\\.)$\n```\n\n### Explanation:\n- `^` - Start of string\n- `(?!.*\\.\\.)` - Negative lookahead: no consecutive periods anywhere\n- `(?!-)` - Negative lookahead: cannot start with hyphen\n- `[a-z0-9.-]{3,63}` - 3-63 characters of lowercase letters, numbers, hyphens, periods\n- `(?<!-)` - Negative lookbehind: cannot end with hyphen\n- `(?<!\\.)` - Negative lookbehind: cannot end with period\n- `$` - End of string\n\n## Solution 2: More Explicit Pattern\n\n```regex\n^[a-z0-9]([a-z0-9.-]{1,61}[a-z0-9])?$(?<!.*\\.\\.)\n```\n\nThis ensures the first and last characters are alphanumeric, with the middle allowing hyphens and periods.\n\n## Implementation Examples\n\n### JavaScript\n```javascript\nconst s3BucketRegex = /^(?!.*\\.\\.)(?!-)[a-z0-9.-]{3,63}(?<!-)(?<